# Solutions — Notebook 1: Receptive fields, one encoder, three tasks

**ML Summer School · Large models · Lecture 2**

---

Worked solutions to the five exercises at the end of Notebook 1.

**Read exercise 2 even if you skip the rest.** It asks you to make the energy
head worse on purpose, and instead it turns up an error in the main notebook's
reasoning. The correction is more useful than the exercise was.

**A warning about the energy metric.** Energy MAE on this task is *noisy*: two
runs differing only in seed can disagree by a factor of two. Where a conclusion
depends on it we average over three seeds and quote the scatter. Where three
seeds are still not enough, we say so rather than reporting the mean as if it
meant something. Classification accuracy and segmentation IoU are far more
stable and single runs are fine for them.

**Runtime:** roughly 12–18 minutes on a Colab T4.

## Setup

Notebook 1's multi-task network, with switches for each exercise: number of
encoder blocks, energy-head pooling, and whether the decoder gets skip
connections.

In [ ]:
# --- setup: make the course package importable (run once per session) ------
# On Colab: clones the repository and installs it.
# Locally inside a checkout: finds it and uses it in place -- no second copy.
# Already importable: does nothing. Safe to re-run either way.
REPO_URL = "https://github.com/drinkingkazu/a3net-lecture2.git"
REPO_DIR = "a3net-lecture2"

import os, subprocess, sys


def _find_checkout(start=None):
    """Walk up from `start` looking for a directory containing mlschool/."""
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, "mlschool", "__init__.py")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


try:
    import mlschool                       # already installed, or already on sys.path
except ModuleNotFoundError:
    root = _find_checkout()               # are we sitting inside the repo already?
    if root is None:                      # no -- fetch it (this is the Colab path)
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                           check=True)
        root = os.path.abspath(REPO_DIR)
        try:                              # nice-to-have; sys.path below is enough
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", root],
                           check=True)
        except subprocess.CalledProcessError:
            print("pip install failed; falling back to sys.path (usually fine)")
    sys.path.insert(0, root)
    import mlschool

import mlschool as ms
print("mlschool", ms.__version__, "from", os.path.dirname(ms.__file__))
print("device:", ms.device())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

train = ms.generate_dataset(4000, seed=0, progress=True)
val   = ms.generate_dataset(1000, seed=1)

CHARGE_SCALE = float(np.percentile(train["image"][train["image"] > 0], 99))
E_MEAN, E_STD = float(train["energy"].mean()), float(train["energy"].std())


def prepare(ds):
    return {"x": torch.tensor(ds["image"])[:, None] / CHARGE_SCALE,
            "cls": torch.tensor(ds["label"]),
            "seg": torch.tensor(ds["seg"]).long(),
            "energy": torch.tensor((ds["energy"] - E_MEAN) / E_STD)}


TR, VA = prepare(train), prepare(val)


def conv_bn_relu(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())


class MultiTaskNet(nn.Module):
    def __init__(self, widths=(16, 32, 64, 96), energy_pool="sum", skips=True):
        super().__init__()
        self.energy_pool, self.skips = energy_pool, skips
        self.blocks = nn.ModuleList()
        cin = 1
        for w in widths:
            self.blocks.append(conv_bn_relu(cin, w))
            cin = w
        deep = widths[-1]
        self.cls_head = nn.Sequential(nn.Linear(deep, 64), nn.ReLU(), nn.Linear(64, 3))
        self.energy_head = nn.Sequential(nn.Linear(deep, 64), nn.ReLU(), nn.Linear(64, 1))
        self.up = nn.ModuleList([
            conv_bn_relu(widths[i] + widths[i - 1] if skips else widths[i], widths[i - 1])
            for i in range(len(widths) - 1, 0, -1)])
        self.seg_out = nn.Conv2d(widths[0], 3, 1)

    def forward(self, x):
        feats = []
        for i, blk in enumerate(self.blocks):
            if i > 0:
                x = F.max_pool2d(x, 2)
            x = blk(x)
            feats.append(x)
        deep = feats[-1]
        cls = self.cls_head(deep.amax(dim=(2, 3)))
        pooled = (deep.sum(dim=(2, 3)) / 100.0 if self.energy_pool == "sum"
                  else deep.mean(dim=(2, 3)))
        energy = self.energy_head(pooled)
        h = deep
        for k, blk in enumerate(self.up):
            skip = feats[-2 - k]
            h = F.interpolate(h, size=skip.shape[-2:], mode="nearest")
            h = blk(torch.cat([h, skip], dim=1) if self.skips else h)
        return cls, self.seg_out(h), energy.squeeze(1)


def focal_loss(logits, target, gamma=2.0):
    ce = F.cross_entropy(logits, target, reduction="none")
    return ((1 - torch.exp(-ce)) ** gamma * ce).mean()


def train_model(widths=(16, 32, 64, 96), energy_pool="sum", skips=True,
                seg_loss="ce", seg_weight=None, tasks=("cls", "seg", "energy"),
                epochs=6, bs=64, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    model = MultiTaskNet(widths, energy_pool, skips).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    w = None if seg_weight is None else seg_weight.to(DEVICE)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(TR["x"]))
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            c, s, e = model(TR["x"][b].to(DEVICE))
            loss = 0.0
            if "cls" in tasks:
                loss = loss + F.cross_entropy(c, TR["cls"][b].to(DEVICE))
            if "seg" in tasks:
                loss = loss + (focal_loss(s, TR["seg"][b].to(DEVICE))
                               if seg_loss == "focal" else
                               F.cross_entropy(s, TR["seg"][b].to(DEVICE), weight=w))
            if "energy" in tasks:
                loss = loss + F.huber_loss(e, TR["energy"][b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()


@torch.no_grad()
def evaluate(model, bs=128):
    ok, seg_pred, e_pred = 0, [], []
    for i in range(0, len(VA["x"]), bs):
        c, s, e = model(VA["x"][i:i + bs].to(DEVICE))
        ok += (c.argmax(1).cpu() == VA["cls"][i:i + bs]).sum().item()
        seg_pred.append(s.argmax(1).cpu()); e_pred.append(e.cpu())
    seg_pred, e_pred = torch.cat(seg_pred), torch.cat(e_pred)
    out = {"cls_acc": ok / len(VA["x"]),
           "energy_mae": (e_pred - VA["energy"]).abs().mean().item() * E_STD,
           "seg": {}}
    for c in range(3):
        p, t = (seg_pred == c), (VA["seg"] == c)
        inter = (p & t).sum().item()
        out["seg"][ms.SEG_NAMES[c]] = {
            "IoU": inter / max((p | t).sum().item(), 1),
            "recall": inter / max(t.sum().item(), 1),
            "precision": inter / max(p.sum().item(), 1)}
    return out


def line(tag, r):
    s = r["seg"]
    print(f"{tag:<30} cls {r['cls_acc']:.3f}   track IoU {s['track']['IoU']:.3f}   "
          f"shower IoU {s['shower']['IoU']:.3f}   E-MAE {r['energy_mae']:.3f}")


base_model = train_model()
baseline = evaluate(base_model)
line("baseline (4 blocks)", baseline)

## Exercise 1 — Break the receptive field

> Retrain the multi-task model with only two encoder blocks. Which of the three
> tasks degrades most, and does the receptive field table explain why?

Two blocks give a receptive field of 16 px instead of 76 px, and a $24\times24$
bottleneck instead of $12\times12$.

Because the energy number is the noisy one, we run the two configurations with
three seeds and report the scatter.

In [ ]:
r2 = evaluate(train_model(widths=(16, 32)))
r3 = evaluate(train_model(widths=(16, 32, 64)))
line("2 encoder blocks (RF 16 px)", r2)
line("3 encoder blocks (RF 36 px)", r3)
line("4 encoder blocks (RF 76 px)", baseline)

# classification and IoU are stable run to run; energy is not, so give it 5 seeds
print("\nenergy MAE over 5 seeds (the only metric that moved):")
energy_runs = {}
for tag, widths in [("4 blocks", (16, 32, 64, 96)), ("2 blocks", (16, 32))]:
    vals = sorted(evaluate(train_model(widths=widths, seed=s))["energy_mae"]
                  for s in range(5))
    energy_runs[tag] = vals
    print(f"  {tag:<10} median {np.median(vals):.3f}   range "
          f"{vals[0]:.3f}-{vals[-1]:.3f}   runs {[f'{v:.3f}' for v in vals]}")

lo, hi = energy_runs["2 blocks"], energy_runs["4 blocks"]
overlap = sum(1 for v in lo if v <= max(hi))
print(f"\n  {overlap} of 5 two-block runs fall inside the four-block range "
      f"-> the effect is real in the median but NOT clean run-to-run.")

### What we measured

| encoder | classification | track IoU | shower IoU |
|---|---|---|---|
| 2 blocks, RF 16 px | 0.996 | 0.889 | 0.873 |
| 3 blocks, RF 36 px | 0.995 | 0.906 | 0.879 |
| 4 blocks, RF 76 px | 0.996 | 0.912 | 0.893 |

energy MAE over 5 seeds:

In one representative run: 4 blocks median 0.076 (range 0.070–0.155), 2 blocks
median 0.179 (range 0.074–0.327). **Read the numbers your own run printed** —
the medians move by tens of percent between runs, and so does the amount of
overlap.

**Classification and segmentation barely move.** Cutting the receptive field from
76 px to 16 px costs 0.000 in classification accuracy and 0.023 in track IoU.
That is the headline, and it is not what the exercise's phrasing leads you to
expect.

**Energy is the only metric that shifts** — the median roughly doubles — **but
the shift is not clean.** The ranges overlap: the luckiest two-block run beats
the unluckiest four-block run, every time we have run this. With five seeds we
can say the median moved; we cannot say that any given pair of runs would show
it. If you need to *establish* this rather than notice it, you need more seeds
than a lecture exercise can afford.

So: which task degrades most? Energy, in the median. Does the receptive-field
table explain why? **Partly, and the part it fails to explain is more useful.**

What it explains: the energy head must *integrate charge over the whole event*,
and calibrating a local activation into a contribution to a total requires
knowing what kind of object you are looking at and how much of it lies outside
your own footprint. Sixteen pixels is not much context for that.

What it does *not* explain is why classification and segmentation are essentially
unaffected. The naive reading of Notebook 1 — bigger receptive field is better —
predicts all three tasks suffer. They do not, because **what separates a track
from a shower here is local texture**: thin and continuous versus wide and
diffuse. Sixteen pixels already resolves that.

The transferable rule is the conditional one: compare your receptive field
against **the physical scale of the feature that distinguishes your classes**,
not against the size of your image. For local texture, small is fine. For an
extensive quantity integrated over a whole event, it is not — and even then, on
this dataset, the effect is smaller and noisier than the story suggests.

## Exercise 2 — Wrong pooling on purpose

> Change the energy head from `sum` to `mean` pooling and retrain. Predict what
> happens to events containing two tracks before you run it.

The prediction from Notebook 1 §3 is confident: energy is **extensive**, two
identical showers deposit twice the energy, mean-pooling normalises that away and
cannot represent it, so it should fail badly on high-multiplicity events.

Let us test it properly. Training-run comparisons of energy MAE are too noisy to
settle this (we tried; two three-seed experiments disagreed about the sign).
Instead, test the property *directly*: build images containing $k$ non-overlapping
copies of the same track, so the true energy is exactly $k\times$ the single-track
energy, and ask each trained model what it predicts.

That is a deterministic test of extensivity, with no training noise in it at all.

In [ ]:
sum_model  = train_model(energy_pool="sum")
mean_model = train_model(energy_pool="mean")

track_idx = np.where(val["label"] == 0)[0][:200]      # single-topology events
base_img = val["image"][track_idx]
OFFSETS = [(0, 0), (0, 32), (32, 0), (32, 32)]


def k_copies(k):
    """k spatially separated copies of each event -> true energy x k."""
    out = np.zeros_like(base_img)
    for dy, dx in OFFSETS[:k]:
        out += np.roll(np.roll(base_img, dy, axis=1), dx, axis=2)
    return torch.tensor(out)[:, None] / CHARGE_SCALE


@torch.no_grad()
def predict_energy(model, x, bs=64):
    return torch.cat([model(x[i:i + bs].to(DEVICE))[2].cpu()
                      for i in range(0, len(x), bs)]) * E_STD + E_MEAN


single = float(val["energy"][track_idx].mean())
print(f"{'k copies':>9}{'true energy':>14}{'sum pool':>12}{'mean pool':>12}")
print("-" * 47)
curves = {"sum": [], "mean": []}
for k in (1, 2, 3, 4):
    xb = k_copies(k)
    ps = predict_energy(sum_model, xb).mean().item()
    pm = predict_energy(mean_model, xb).mean().item()
    curves["sum"].append(ps); curves["mean"].append(pm)
    print(f"{k:>9}{single * k:>14.3f}{ps:>12.3f}{pm:>12.3f}")

print("\nratio to the k=1 prediction  (an extensive head should give k):")
for p, v in curves.items():
    print(f"  {p:>5} pool: " + "   ".join(f"k={k}: {x / v[0]:.2f}"
                                          for k, x in zip((1, 2, 3, 4), v)))

fig, ax = plt.subplots(figsize=(5.6, 3.8))
ks = [1, 2, 3, 4]
ax.plot(ks, [single * k for k in ks], "k--", label="true energy")
ax.plot(ks, curves["sum"], "o-", label="sum pooling")
ax.plot(ks, curves["mean"], "s-", label="mean pooling")
ax.set_xlabel("number of copies of the event in the image")
ax.set_ylabel("predicted energy"); ax.set_xticks(ks)
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.show()

### The prediction is wrong, and so was the notebook

| k | true | sum pool | mean pool |
|---|---|---|---|
| 1 | 0.550 | 0.467 | 0.448 |
| 2 | 1.100 | 0.883 | 0.883 |
| 3 | 1.650 | 1.301 | 1.336 |
| 4 | 2.199 | 1.632 | 1.689 |

Ratios to $k=1$ — sum: 1.00, 1.89, 2.79, 3.49. Mean: 1.00, 1.97, 2.98, 3.77.

**Both are extensive, and they are indistinguishable.** Mean pooling tracks the
linear growth at least as well as sum pooling does; on this run it is very
slightly *closer* to linear, which is noise rather than a result.

Here is why, and it is embarrassingly simple once you see it. Our input is a
fixed $96\times96$ image, so the bottleneck feature map is always
$6\times6 = 36$ cells. Therefore

$$\text{mean} \;=\; \frac{1}{36}\sum_{\text{cells}} \;=\; \frac{\text{sum}}{36}.$$

For a **fixed-size** input, global mean pooling *is* global sum pooling divided
by a constant — and the linear layer that follows absorbs the constant during
training. The two heads are the same model up to a rescaling of one weight
matrix. There is no prior being expressed at all. This is an argument from the
architecture, not from a measurement, which is why it does not care about seeds.

**Notebook 1 §3 overstated this**, and the notebook has since been corrected: the
claim that "only sum-pooling can represent an extensive quantity" is false for a
dense CNN on fixed-size images.

**When does the distinction become real?** When the number of pooled elements
*varies between examples*. That is exactly the point-cloud setting of Notebook 3,
where each event has a different number of hits, so dividing by that count throws
multiplicity information away. The argument was sound; it was applied in the
wrong place.

**A second thing worth noticing:** both heads under-predict at $k=4$ (3.5 and 3.8
rather than 4.0). Neither model ever saw a four-track event — the training data
tops out at two tracks plus a shower — so by $k=4$ they are extrapolating outside
their training distribution and are beginning to saturate. That is a useful
reminder in its own right: an extensive head is only extensive over the range you
trained it on.

Two general lessons, both worth more than the original claim:

- **An architectural "prior" that a downstream linear layer can undo is not a
  prior.** Before asserting that a design choice encodes an assumption, check
  whether the network can trivially invert it.
- **The right test for a scaling property is a constructed input, not a metric.**
  One deterministic superposition test settled in seconds what a dozen noisy
  training runs could not.

## Exercise 3 — Remove the U-Net skips

> Feed only the upsampled features to each decoder block. Classification should
> barely move; segmentation should get visibly blurry. Why?

In [ ]:
no_skip = evaluate(train_model(skips=False))
line("with U-Net skips", baseline)
line("without U-Net skips", no_skip)

print(f"\n{'class':<12}{'IoU with':>10}{'IoU without':>13}")
for c in ["background", "track", "shower"]:
    print(f"{c:<12}{baseline['seg'][c]['IoU']:>10.3f}"
          f"{no_skip['seg'][c]['IoU']:>13.3f}")

In [ ]:
model_ns = train_model(skips=False)          # trained outside the no_grad block!
idx = [3, 11, 19, 27]
xb = VA["x"][idx].to(DEVICE)
with torch.no_grad():
    pred_with = base_model(xb)[1].argmax(1).cpu()
    pred_without = model_ns(xb)[1].argmax(1).cpu()

from matplotlib.colors import ListedColormap
cmap = ListedColormap(ms.SEG_COLORS)
fig, axes = plt.subplots(3, len(idx), figsize=(2.3 * len(idx), 7))
for k, i in enumerate(idx):
    axes[0, k].imshow(VA["seg"][i], cmap=cmap, vmin=0, vmax=2, origin="lower")
    axes[1, k].imshow(pred_with[k], cmap=cmap, vmin=0, vmax=2, origin="lower")
    axes[2, k].imshow(pred_without[k], cmap=cmap, vmin=0, vmax=2, origin="lower")
    for r in range(3):
        axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
for r, name in enumerate(["truth", "with skips", "without skips"]):
    axes[r, 0].set_ylabel(name, fontsize=9)
fig.tight_layout(); plt.show()

### What we measured

| | classification | track IoU | shower IoU |
|---|---|---|---|
| with skips | 0.996 | **0.912** | **0.893** |
| without skips | 0.992 | **0.040** | **0.521** |

Predicted, and the magnitude is startling: **track IoU falls from 0.91 to 0.04**
— the track class is essentially gone — while classification loses 0.004.

The reason is a counting argument. Without skips, the decoder's only input is the
96-channel $12\times12$ bottleneck. To label a $96\times96$ image it must invent
64 pixels of detail from each bottleneck cell, and four max-pools destroyed the
information needed to do it. Upsampling can place a blurred blob roughly
correctly — which is why shower IoU (0.52) survives far better than track IoU
(0.04). **A shower *is* a blob; a track is a one-pixel-wide line**, and a line
whose position is wrong by two pixels overlaps the truth almost nowhere. IoU is
merciless about that.

Classification is unaffected because it only ever consumed the bottleneck. It
never needed the high-resolution detail, so removing the path that carries it
costs nothing.

This is the cleanest illustration of the two-jobs distinction from Notebook 1:
**the encoder decides *what*; the skips decide *where*.**

## Exercise 4 — Focal loss

> Try focal loss ($\mathcal{L} = -(1-p_t)^\gamma \log p_t$, $\gamma \approx 2$)
> and compare its precision/recall trade-off against inverse-frequency weighting.

Recall the main notebook's finding: inverse-frequency weighting made things
*worse* here, because the classes are not confusable once the model has a
receptive field. Focal loss attacks a different quantity — it down-weights
examples the model already gets right, concentrating the gradient on genuinely
hard pixels, rather than up-weighting a whole class.

In [ ]:
counts = np.bincount(train["seg"].ravel(), minlength=3)
frac = counts / counts.sum()
inv = 1.0 / frac
class_weights = torch.tensor(inv / inv.mean(), dtype=torch.float32)

runs = {"plain cross-entropy": baseline,
        "focal loss (gamma=2)": evaluate(train_model(seg_loss="focal")),
        "inverse-frequency CE": evaluate(train_model(seg_weight=class_weights))}

for cls in ["track", "shower"]:
    print(f"\n{cls}:")
    print(f"  {'loss':<24}{'IoU':>8}{'recall':>10}{'precision':>12}")
    for tag, r in runs.items():
        d = r["seg"][cls]
        print(f"  {tag:<24}{d['IoU']:>8.3f}{d['recall']:>10.3f}{d['precision']:>12.3f}")

### What we measured

**track**

| loss | IoU | recall | precision |
|---|---|---|---|
| plain cross-entropy | **0.912** | 0.928 | 0.981 |
| focal (γ=2) | 0.909 | 0.952 | 0.952 |
| inverse-frequency | 0.758 | 0.987 | 0.765 |

**shower**

| loss | IoU | recall | precision |
|---|---|---|---|
| plain cross-entropy | **0.893** | 0.937 | 0.951 |
| focal (γ=2) | 0.863 | 0.902 | 0.952 |
| inverse-frequency | 0.761 | 0.972 | 0.778 |

Focal loss is **essentially indistinguishable from plain cross-entropy** on IoU,
and both are far better than inverse-frequency weighting.

That is the right outcome, and it follows from the main notebook's diagnosis.
Focal loss is a remedy for a large population of *easy* examples swamping the
gradient. Here the easy population is background pixels that are exactly zero;
the model drives their loss so close to zero that the $(1-p_t)^\gamma$ factor has
almost nothing left to suppress. Applied to a problem that does not exist, a
remedy is a no-op.

Now the part the exercise actually asks about — the **trade-off**, which is
visible even though the IoU is not:

- **Inverse-frequency weighting** moves the dial hard: track recall 0.928 → 0.987,
  precision 0.981 → 0.765. It makes the model eager, and pays for every extra
  true track with several false ones.
- **Focal loss** moves it gently in the same direction: recall 0.928 → 0.952,
  precision 0.981 → 0.952. Roughly a wash in IoU, but a genuinely different
  operating point.

So the ordering is: if you need recall at almost any cost, weighting is the blunt
instrument that delivers it. If you want a small, controlled shift towards hard
pixels, focal loss gives you that. If you want the best IoU on *this* dataset, do
nothing at all — which was the main notebook's point.

## Exercise 5 — Single-task baselines

> Train classification alone, with no segmentation or energy loss. Is the
> multi-task model better? By how much?

In [ ]:
# On the full training set this comparison is a dead end -- see the discussion.
print("4000 training events (the notebook's setting)")
for tag, tasks in [("classification only", ("cls",)), ("multi-task", None)]:
    v = sorted(evaluate(train_model(tasks=tasks or ("cls", "seg", "energy"),
                                    seed=s))["cls_acc"] for s in range(3))
    print(f"  {tag:<22} median {np.median(v):.3f}   runs {[f'{x:.3f}' for x in v]}")

In [ ]:
# Give the comparison some headroom: train on 400 events instead of 4000.
TR_FULL = TR
TR = {k: v[:400] for k, v in TR_FULL.items()}

print("400 training events (where there is room to improve)")
small = {}
for tag, tasks in [("classification only", ("cls",)),
                   ("multi-task", ("cls", "seg", "energy"))]:
    v = sorted(evaluate(train_model(tasks=tasks, epochs=12, seed=s))["cls_acc"]
               for s in range(3))
    small[tag] = v
    print(f"  {tag:<22} median {np.median(v):.3f}   runs {[f'{x:.3f}' for x in v]}")

TR = TR_FULL          # restore
gain = np.median(small["multi-task"]) - np.median(small["classification only"])
print(f"\n  multi-task advantage at 400 events: {gain:+.3f} accuracy")

### What we measured

**On the full 4 000 events there is no measurable difference.** Both the
classification-only and the multi-task model return a median of ~0.996, and both
occasionally produce a worse seed. We ran this comparison several times while
preparing these solutions and the ordering flipped between runs.

That is not a failure of the experiment; it is the experiment telling you the
question is badly posed **at this dataset size**. The single-task model already
gets 99.6 % — there is no room for an auxiliary task to add anything. Any
"improvement" you measure here is seed noise, and if you had run it once and
reported the direction you happened to get, you would have published noise.

**So we gave it headroom**: the same comparison with 400 training events, where
the classifier is genuinely short of signal. Now the effect is unambiguous.

| 400 events | median accuracy | individual runs |
|---|---|---|
| classification only | 0.893 | 0.370, 0.893, 0.947 |
| multi-task | **0.981** | 0.961, 0.981, 0.982 |

**+0.088 in the median, and — look at the individual runs — a collapse in the
variance.** The single-task model produced one catastrophic seed (0.370, barely
above chance); the multi-task model's three runs span 0.02. When labels are
scarce, the auxiliary supervision is not just worth a few points of accuracy, it
is what makes training *reliable*.

The mechanism it is testing is supervision density. A classification label is
*one number per event*. The segmentation labels give 9 216 labelled pixels per
event, and they force the encoder to learn where the track is and where the
shower is — precisely what the classifier needs. When labels are scarce, that
extra signal is worth a great deal; when you already have enough, it is worth
nothing.

Three things to take away:

- **You cannot demonstrate a regulariser on a task you have already solved.**
  Saturated metrics hide every effect. Design the comparison in the regime where
  the quantity can move.
- **Multi-task learning is not free.** The tasks compete for capacity and the loss
  weights are real hyperparameters — the main notebook shows a case where
  over-weighting segmentation degraded the energy head.
- **If you want a useful auxiliary task, pick one with dense labels.** A second
  event-level label supplies almost no extra signal; a per-pixel or per-hit label
  supplies thousands of constraints per event.